In [1]:
from pathlib import Path

import open3d as o3d
import numpy as np
import torch
from torch.utils.data import DataLoader

from plant_shape_analysis.models.siren import PointCloudSiren

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Load and visualize a sample leaf point cloud
sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/1_maize_control_plant1_D00_leaf2_dense.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))
leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
leaf_np = np.asarray(leaf_pcd.points)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")
o3d.visualization.draw_geometries([leaf_pcd])

Loaded leaf point cloud with 34789 points.


### Visualize current sampling strategy

In [3]:
# Plot off-surface points sampled around the leaf
sampling_strategy = "gaussian"  # "gaussian"
leaf_dataset = PointCloudSiren(point_cloud, len(point_cloud), sampling_strategy=sampling_strategy, k_neighbors=200)
dataloader = DataLoader(leaf_dataset, shuffle=True, batch_size=1, pin_memory=True, num_workers=0)
for data in dataloader:
    print(data)
    break

Off surface sampling strategy:  gaussian
[{'coords': tensor([[[ 0.0746, -0.0623,  0.3353],
         [-0.0163,  0.0305, -0.5751],
         [-0.0649, -0.0778, -0.4072],
         ...,
         [-0.1148,  0.0912,  0.7327],
         [ 0.0483,  0.1284,  0.0907],
         [ 0.0885, -0.1389,  0.4582]]])}, {'sdf': tensor([[[ 0.],
         [ 0.],
         [ 0.],
         ...,
         [-1.],
         [-1.],
         [-1.]]]), 'normals': tensor([[[ 0.9098, -0.0591,  0.4109],
         [-0.6902,  0.7221, -0.0468],
         [-0.9776, -0.2059, -0.0434],
         ...,
         [-1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000]]])}]


In [4]:
points = data[0]['coords'].squeeze()
if sampling_strategy in ["uniform", "gaussian"]:
    surface_points, off_surface_points = points[:len(leaf_np)], points[len(leaf_np):]
elif sampling_strategy == "mixed":
    surface_points, off_surface_points = points[:len(leaf_np)], points[len(leaf_np):]
    off_surface_points_local, off_surface_points_global = off_surface_points[:len(leaf_np)//2], off_surface_points[len(leaf_np)//2:]

In [ ]:
# Surface points
o3d.visualization.draw_geometries([o3d.geometry.PointCloud(o3d.utility.Vector3dVector(surface_points.numpy()))])

In [5]:
# Off-surface points global
o3d.visualization.draw_geometries([o3d.geometry.PointCloud(o3d.utility.Vector3dVector(off_surface_points.numpy()))])

In [ ]:
# Off-surface points local
o3d.visualization.draw_geometries([o3d.geometry.PointCloud(o3d.utility.Vector3dVector(off_surface_points_local.numpy()))])

Points are too far away from the surface!

### IGR sampling strategy

In [6]:
local_sigma = 0.02
global_sigma = 1.8
igr_close = surface_points + (torch.randn_like(surface_points) * local_sigma)
igr_far = (torch.rand(1, len(leaf_np) // 8, 3) * (global_sigma * 2)) - global_sigma

In [7]:
points = np.vstack([surface_points, igr_close, igr_far.squeeze().numpy()])
points = np.vstack([surface_points, igr_close])
points = np.vstack([igr_close])

In [8]:
points.shape

(34789, 3)

In [9]:
o3d.visualization.draw_geometries([o3d.geometry.PointCloud(o3d.utility.Vector3dVector(points))])

Close points are much more fine grained, in current sampling strategy the gaussian points are too far away from the surface